# Enterprise RAG — Hands-On, Part 4 of 11: Chunking and ingestion

*Split from `02-hands-on.ipynb` for focused reading — same content, one phase at a time. The
"Setup" cell below re-derives whatever state earlier parts would have produced, so this notebook
runs standalone; you do not need to run the other parts first.*

**Prerequisites:** `OPENAI_API_KEY` in the repo-root `.env`, and `python scripts/ingest.py` already
run (the setup cell below will build the index for you if it is missing).

**Series:** [1. The corpus and its permissions](part01-corpus-and-permissions.ipynb) · [2. The policy engine](part02-policy-engine.ipynb) · [3. Compiling the policy into a database filter](part03-compiling-policy-to-filter.ipynb) · [4. Chunking and ingestion](part04-chunking-and-ingestion.ipynb) · [5. Why hybrid search, demonstrated](part05-hybrid-search.ipynb) · [6. Query transformation](part06-query-transformation.ipynb) · [7. Reranking](part07-reranking.ipynb) · [8. The full graph](part08-full-graph.ipynb) · [9. Attacking it](part09-attacking-it.ipynb) · [10. Evaluation](part10-evaluation.ipynb) · [11. Observability, and what to take away](part11-observability-and-takeaways.ipynb)

---


In [ ]:
import sys, json, textwrap
from pathlib import Path

# The package lives in src/ - add it to the path so this notebook runs from anywhere.
ROOT = Path.cwd()
while not (ROOT / "src" / "enterprise_rag").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from enterprise_rag.config import SETTINGS

print("project root :", ROOT)
print("corpus       :", SETTINGS.corpus_dir.relative_to(ROOT))
print("api key      :", "found" if SETTINGS.has_api_key else "MISSING - check .env")
print("embed model  :", SETTINGS.embedding_model)
print("chat model   :", SETTINGS.fast_model)

### Setup — recap of state from earlier parts


In [ ]:
from enterprise_rag.ingest.loader import load_corpus

docs = load_corpus()

---
# Part 4 - Chunking and ingestion

Chunks split on markdown headings first, then pack to a target size. Every chunk **inherits its
parent's ACL attributes** - that denormalisation is what makes a single-pass pre-filter possible.

Ingestion does two independent things with each validated document: it chunks and embeds it into the
vector index (below), **and** it writes one row per document into a separate ACL catalog (SQLite) -
see `ingest/catalog.py`. The catalog write does not depend on chunking or embedding at all, which is
the whole point: a later access-rule change only ever needs a write to that one row, never a
re-chunk or a re-embed.

In [ ]:
from enterprise_rag.ingest.chunker import chunk_document

doc = next(d for d in docs if d.attrs.doc_id == "CT-VTX-001")
chunks = chunk_document(doc)
print(f"{doc.attrs.doc_id} -> {len(chunks)} chunks\n")
for c in chunks[:4]:
    print(f"[{c.chunk_id}] section={c.section!r}  ({len(c.text)} chars)")
    print(textwrap.indent(textwrap.fill(c.text[:200], 88), "    "), "\n")

In [ ]:
# The service-credit tiers survive as ONE coherent chunk - the whole point of
# structure-aware splitting.
credits = next(c for c in chunks if "credit" in c.section.lower())
print(credits.text)

In [ ]:
# Every chunk carries the parent's permissions, and metadata is flattened to
# Chroma-compatible scalars (note the grp__* boolean columns).
print(json.dumps(credits.to_metadata(), indent=2))

In [ ]:
# Build the index if it is not already there.
from enterprise_rag.ingest.store import collection_stats
try:
    stats = collection_stats("meridian")
    assert stats["chunks"] > 0
    print("index already built:", stats)
except Exception:
    from enterprise_rag.ingest.pipeline import ingest
    print("building index...")
    print(ingest().render())

---

**◀ Previous:** [3. Compiling the policy into a database filter](part03-compiling-policy-to-filter.ipynb)

**Next ▶:** [5. Why hybrid search, demonstrated](part05-hybrid-search.ipynb)
